# Eliminating Relational Data Fan-Out with BigQuery Graph Measures

### A Step-by-Step Tutorial on Building a Governed Semantic Layer for Precise Financial & Resource Reporting

---
**Measures** in BigQuery Graph solves data fan-out natively. By embedding the `MEASURE()` keyword directly into the node definition of a Property Graph, you lock the aggregation logic strictly to the parent node's primary key (e.g., `dept_id`).

In this tutorial, you will:
1. **Setup** a multi-dimensional university dataset in BigQuery.
2. **Demonstrate the Fan-Out Trap** using standard SQL joins.
3. **Build a BigQuery Property Graph** with native **Measures**.
4. **Flatten & Query the Graph** using `GRAPH_EXPAND` and `AGG()`.
5. **Solve Many-to-Many (M:N) Relationships** for measures.

## 1. Setup and Configuration
Specify your Google Cloud Project ID and the Region where you want to deploy the BigQuery dataset.

In [ ]:
# Configuration - Replace with your Google Cloud Project ID
GCP_PROJECT_ID = "YOUR_PROJECT_ID" # <-- UPDATE THIS
REGION = "us-central1"
BQ_DATASET_ID = "university_measures"

from google.cloud import bigquery

# Initialize BigQuery client
bq_client = bigquery.Client(project=GCP_PROJECT_ID)
print(f"BigQuery client initialized for project: {GCP_PROJECT_ID} in region: {REGION}")

## 2. Create Dataset and Relational Tables
We will create a dataset and three tables representing our relational schema: `College`, `Department`, and `Course`.

In [ ]:
# Create the BigQuery Dataset
dataset_ref = bigquery.Dataset(f"{GCP_PROJECT_ID}.{BQ_DATASET_ID}")
dataset_ref.location = "US"
dataset = bq_client.create_dataset(dataset_ref, exists_ok=True)

# Define DDL queries for creating the relational tables
create_tables_query = f"""
CREATE OR REPLACE TABLE `{GCP_PROJECT_ID}.{BQ_DATASET_ID}.College` (
  college_id INT64 PRIMARY KEY NOT ENFORCED,
  college_name STRING
);

CREATE OR REPLACE TABLE `{GCP_PROJECT_ID}.{BQ_DATASET_ID}.Department` (
  dept_id INT64 PRIMARY KEY NOT ENFORCED,
  dept_name STRING,
  college_id INT64,
  budget FLOAT64,
  FOREIGN KEY (college_id) REFERENCES `{GCP_PROJECT_ID}.{BQ_DATASET_ID}.College`(college_id) NOT ENFORCED
);

CREATE OR REPLACE TABLE `{GCP_PROJECT_ID}.{BQ_DATASET_ID}.Course` (
  course_id INT64 PRIMARY KEY NOT ENFORCED,
  course_name STRING,
  dept_id INT64,
  credits INT64,
  FOREIGN KEY (dept_id) REFERENCES `{GCP_PROJECT_ID}.{BQ_DATASET_ID}.Department`(dept_id) NOT ENFORCED
);
"""
bq_client.query(create_tables_query).result()
print("Relational tables created successfully.")

## 3. Ingest and Map Relationships
Let's insert some sample data. Notice the CS department offers two courses. In a flat join, its budget will duplicate.

In [ ]:
insert_data_query = f"""
INSERT INTO `{GCP_PROJECT_ID}.{BQ_DATASET_ID}.College` (college_id, college_name)
VALUES (101, 'College of Engineering'), (102, 'College of Arts');

INSERT INTO `{GCP_PROJECT_ID}.{BQ_DATASET_ID}.Department` (dept_id, dept_name, college_id, budget)
VALUES (1001, 'Computer Science', 101, 500000.0),
       (1002, 'Mechanical Engineering', 101, 400000.0),
       (1003, 'Fine Arts', 102, 200000.0);

INSERT INTO `{GCP_PROJECT_ID}.{BQ_DATASET_ID}.Course` (course_id, course_name, dept_id, credits)
VALUES (1, 'Intro to CS', 1001, 3),
       (2, 'Data Structures', 1001, 4),
       (3, 'Thermodynamics', 1002, 3);
"""
bq_client.query(insert_data_query).result()
print("Sample data inserted successfully.")

## 4. The Legacy "Data Fan-Out Trap"
Watch what happens to the CS department's budget when we execute a standard SQL join.

In [ ]:
aggregate_fan_out_query = f"""
SELECT
  col.college_name,
  SUM(dept.budget) AS incorrect_total_budget
FROM `{GCP_PROJECT_ID}.{BQ_DATASET_ID}.Course` c
LEFT JOIN `{GCP_PROJECT_ID}.{BQ_DATASET_ID}.Department` dept ON c.dept_id = dept.dept_id
LEFT JOIN `{GCP_PROJECT_ID}.{BQ_DATASET_ID}.College` col ON dept.college_id = col.college_id
GROUP BY col.college_name;
"""
df_agg_fan_out = bq_client.query(aggregate_fan_out_query).to_dataframe()
print("--- Aggregation with Fan-Out (SUM(budget)) ---")
print(df_agg_fan_out)

## 5. Define the Measure
We declare a Property Graph. On `Department`, we set `MEASURE(SUM(budget)) AS total_budget`. This binds the aggregation to the `Department` node's primary key.

In [ ]:
create_graph_query = f"""
CREATE OR REPLACE PROPERTY GRAPH `{GCP_PROJECT_ID}.{BQ_DATASET_ID}.SchoolGraph`
  NODE TABLES (
    `{GCP_PROJECT_ID}.{BQ_DATASET_ID}.College` AS College
      KEY(college_id)
      PROPERTIES(college_id, college_name),
    `{GCP_PROJECT_ID}.{BQ_DATASET_ID}.Department` AS Department
      KEY(dept_id)
      PROPERTIES(dept_id, dept_name, college_id,
        budget,
        MEASURE(SUM(budget)) AS total_budget),
    `{GCP_PROJECT_ID}.{BQ_DATASET_ID}.Course` AS Course
      KEY(course_id)
      PROPERTIES(course_id, course_name, credits, dept_id,
        MEASURE(COUNT(course_id)) AS course_count)
  )
  EDGE TABLES (
    `{GCP_PROJECT_ID}.{BQ_DATASET_ID}.Department` AS CollegeDept
      SOURCE KEY (college_id) REFERENCES College (college_id)
      DESTINATION KEY (dept_id) REFERENCES Department (dept_id),
    `{GCP_PROJECT_ID}.{BQ_DATASET_ID}.Course` AS DeptCourse
      SOURCE KEY (dept_id) REFERENCES Department (dept_id)
      DESTINATION KEY (course_id) REFERENCES Course (course_id)
  );
"""
bq_client.query(create_graph_query).result()
print("Property Graph created.")

## 6. Aggregating with Absolute Precision
We use `GRAPH_EXPAND` to flatten the view, and `AGG()` to call our defined measure. This guarantees perfect math with no duplication.

In [ ]:
precise_agg_query = f"""
SELECT
  College_college_name,
  AGG(Department_total_budget) AS precise_college_budget,
  AGG(Course_course_count) AS total_courses
FROM GRAPH_EXPAND("{GCP_PROJECT_ID}.{BQ_DATASET_ID}.SchoolGraph")
GROUP BY College_college_name;
"""
df_precise = bq_client.query(precise_agg_query).to_dataframe()
print("--- Governed Aggregation using AGG() ---")
print(df_precise)

## 7. Cleanup

In [ ]:
# Clean up by deleting the dataset
cleanup_query = f"DROP SCHEMA IF EXISTS `{GCP_PROJECT_ID}.{BQ_DATASET_ID}` CASCADE;"
bq_client.query(cleanup_query).result()
print("Environment cleaned up.")